# Temporal transaction features

In this notebook I build historical features from the synthetic transaction data

## Objectives

- Order transactions by customer and timestamp
- Recover each customer's previous transaction
- Measure time between transactions
- Count recent transactions in seven day windows
- Calculate historical averages over 30 and 90 days
- Calculate expanding historical averages
- Aggregate customer activity by month
- Avoid using the current transaction when calculating historical features

The resulting variables will be used during statistical anomaly detection

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent

else:
    project_root = current_directory

In [3]:
synthetic_sample_directory = (
    project_root
    / "data"
    / "synthetic"
    / "sample"
)

synthetic_sample_directory

WindowsPath('c:/Users/Focus/Documents/GitHub/transaction-intelligence/data/synthetic/sample')

In [4]:
transactions_path = (
    synthetic_sample_directory
    / "synthetic_transactions_1000.csv"
)

accounts_path = (
    synthetic_sample_directory
    / "synthetic_accounts.csv"
)

customers_path = (
    synthetic_sample_directory
    / "synthetic_customers.csv"
)

merchants_path = (
    synthetic_sample_directory
    / "synthetic_merchants.csv"
)

In [5]:
file_validation = pd.Series(
    {
        "transactions": transactions_path.exists(),
        "accounts": accounts_path.exists(),
        "customers": customers_path.exists(),
        "merchants": merchants_path.exists(),
    },
    name="exists",
)
file_validation

transactions    True
accounts        True
customers       True
merchants       True
Name: exists, dtype: bool

In [6]:
transactions = pd.read_csv(
    transactions_path, parse_dates=["timestamp"],
)

accounts = pd.read_csv(
    accounts_path,
    parse_dates=["opening_date"],
)

customers = pd.read_csv(
    customers_path,
    parse_dates=["signup_date"],
)

merchants = pd.read_csv(
    merchants_path,
)

In [7]:
print("Transactions:", transactions.shape)
print("Accounts:", accounts.shape)
print("Customers:", customers.shape)
print("Merchants:", merchants.shape)

Transactions: (1000, 10)
Accounts: (119, 6)
Customers: (100, 9)
Merchants: (80, 6)


In [8]:
# Transactions have account_id but do not have directly customer_id

accounts_lookup = accounts[["account_id", "customer_id"]].copy()

In [9]:
customers_lookup = customers[
    [
        "customer_id",
        "country",
        "customer_segment",
        "typical_amount",
        "preferred_hour",
        "international_probability"
    ]
].copy()

In [10]:
customers_lookup = customers_lookup.rename(
    columns={
            "country": "customer_country",
    }
)

In [11]:
merchants_lookup = merchants[
    [
        "merchant_id",
        "category",
        "country",
        "online"
    ]
].copy()

In [12]:
# I add customer_id to each transaction through account_id
transactions = transactions.merge(
    accounts_lookup,
    on = "account_id",
    how = "left",
    validate = "many_to_one"
)

In [13]:
# I also add personal info of each customer profile
transactions = transactions.merge(
    customers_lookup,
    on = "customer_id",
    how = "left",
    validate = "many_to_one"
)

In [14]:
# Merchant info
transactions = transactions.merge(
    merchants_lookup,
    on = "merchant_id",
    how = "left",
    validate = "many_to_one"
)

transactions = transactions.rename(
    columns = {"category" : "merchant_category"}
)

In [15]:
transactions.shape

(1000, 19)

In [16]:
"""
To create the actual amount:
    Completed purchase: positive amount
    Full refund: negative amount
    Declined transaction: zero
"""
transactions["signed_amount"] = np.where(
    transactions["transaction_type"].eq("Refund"),
    - transactions["amount"],
    transactions["amount"]
)

In [17]:
# Effective amount
transactions["effective_amount"] = np.where(
    transactions["status"].eq("Completed"),
    transactions["signed_amount"],
    0.0
)

In [18]:
# Chronological order of transactions / customer
transactions = (
    transactions
    .sort_values(
        [
            "customer_id",
            "timestamp",
            "transaction_id",
        ]
    )
    .reset_index(
        drop=True
    )
)

In [19]:
transactions[
    [
        "customer_id",
        "transaction_id",
        "timestamp",
        "amount",
    ]
].head(15)

,customer_id,transaction_id,timestamp,amount
0,C00001,T0000715,2025-01-10 13:13:00,30.51
1,C00001,T0000201,2025-01-20 12:41:00,27.45
2,C00001,T0000850,2025-01-23 09:41:00,105.28
3,C00001,T0000193,2025-01-29 10:54:00,28.29
4,C00001,T0000803,2025-02-10 09:47:00,24.60
5,C00001,T0000818,2025-02-13 09:03:00,56.49
6,C00001,T0000737,2025-02-16 08:34:00,15.14
7,C00001,T0000396,2025-02-27 08:44:00,12.31
8,C00001,T0000279,2025-03-11 10:45:00,27.02
9,C00001,T0000536,2025-03-27 14:59:00,14.10


In [20]:
# Number transactions / customer
transactions["customer_transaction_number"] = (transactions.groupby("customer_id").cumcount() + 1)

In [21]:
# Timestamp, amount and category of previous transaction (of the same customer)
transactions["previous_timestamp"] = (transactions.groupby("customer_id")["timestamp"].shift(1))
transactions["previous_amount"] = (transactions.groupby("customer_id")["amount"].shift(1))
transactions["previous_merchant_category"] = (transactions.groupby("customer_id")["merchant_category"].shift(1))

In [22]:
transactions.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status,is_injected_anomaly,anomaly_type,...,international_probability,merchant_category,country,online,signed_amount,effective_amount,customer_transaction_number,previous_timestamp,previous_amount,previous_merchant_category
0,T0000715,A00001,M00060,2025-01-10 13:13:00,30.51,EUR,Purchase,Completed,False,none,...,0.321,Entertainment,Italy,True,30.51,30.51,1,NaT,NaN,NaN
1,T0000201,A00001,M00017,2025-01-20 12:41:00,27.45,EUR,Purchase,Completed,False,none,...,0.321,Books,Spain,True,27.45,27.45,2,2025-01-10 13:13:00,30.51,Entertainment
2,T0000850,A00001,M00057,2025-01-23 09:41:00,105.28,EUR,Purchase,Completed,False,none,...,0.321,Entertainment,Spain,True,105.28,105.28,3,2025-01-20 12:41:00,27.45,Books
3,T0000193,A00001,M00001,2025-01-29 10:54:00,28.29,EUR,Purchase,Completed,False,none,...,0.321,Groceries,Spain,False,28.29,28.29,4,2025-01-23 09:41:00,105.28,Entertainment
4,T0000803,A00001,M00001,2025-02-10 09:47:00,24.60,EUR,Purchase,Completed,False,none,...,0.321,Groceries,Spain,False,24.60,24.60,5,2025-01-29 10:54:00,28.29,Groceries


In [23]:
transactions[
    [
        "customer_id",
        "timestamp",
        "previous_timestamp",
        "amount",
        "previous_amount",
        "merchant_category",
        "previous_merchant_category",
    ]
].head(15)

,customer_id,timestamp,previous_timestamp,amount,previous_amount,merchant_category,previous_merchant_category
0,C00001,2025-01-10 13:13:00,NaT,30.51,NaN,Entertainment,NaN
1,C00001,2025-01-20 12:41:00,2025-01-10 13:13:00,27.45,30.51,Books,Entertainment
2,C00001,2025-01-23 09:41:00,2025-01-20 12:41:00,105.28,27.45,Entertainment,Books
3,C00001,2025-01-29 10:54:00,2025-01-23 09:41:00,28.29,105.28,Groceries,Entertainment
4,C00001,2025-02-10 09:47:00,2025-01-29 10:54:00,24.60,28.29,Groceries,Groceries
5,C00001,2025-02-13 09:03:00,2025-02-10 09:47:00,56.49,24.60,Groceries,Groceries
6,C00001,2025-02-16 08:34:00,2025-02-13 09:03:00,15.14,56.49,Books,Groceries
7,C00001,2025-02-27 08:44:00,2025-02-16 08:34:00,12.31,15.14,Books,Books
8,C00001,2025-03-11 10:45:00,2025-02-27 08:44:00,27.02,12.31,Entertainment,Books
9,C00001,2025-03-27 14:59:00,2025-03-11 10:45:00,14.10,27.02,Restaurants,Entertainment


In [24]:
# Time since previous transaction
transactions["time_since_previous_transaction"] = (transactions.groupby("customer_id")["timestamp"].diff())

In [25]:
transactions["hours_since_previous_transaction"] =  transactions["time_since_previous_transaction"].dt.total_seconds()/3600
transactions["days_since_previous_transaction"] =  transactions["hours_since_previous_transaction"]/24

In [26]:
transactions[
    [
        "customer_id",
        "timestamp",
        "previous_timestamp",
        "hours_since_previous_transaction",
        "days_since_previous_transaction",
    ]
].head(15)

,customer_id,timestamp,previous_timestamp,hours_since_previous_transaction,days_since_previous_transaction
0,C00001,2025-01-10 13:13:00,NaT,NaN,NaN
1,C00001,2025-01-20 12:41:00,2025-01-10 13:13:00,239.466667,9.977778
2,C00001,2025-01-23 09:41:00,2025-01-20 12:41:00,69.000000,2.875000
3,C00001,2025-01-29 10:54:00,2025-01-23 09:41:00,145.216667,6.050694
4,C00001,2025-02-10 09:47:00,2025-01-29 10:54:00,286.883333,11.953472
5,C00001,2025-02-13 09:03:00,2025-02-10 09:47:00,71.266667,2.969444
6,C00001,2025-02-16 08:34:00,2025-02-13 09:03:00,71.516667,2.979861
7,C00001,2025-02-27 08:44:00,2025-02-16 08:34:00,264.166667,11.006944
8,C00001,2025-03-11 10:45:00,2025-02-27 08:44:00,290.016667,12.084028
9,C00001,2025-03-27 14:59:00,2025-03-11 10:45:00,388.233333,16.176389


In [27]:
# Temporal windows 
transactions_previous_7d = (transactions.groupby("customer_id", sort = False)
                        .rolling(window = "7D", on = "timestamp", closed = "left")
                        ["transaction_id"].count().fillna(0).astype(int).to_numpy()
)

In [28]:
transactions["transactions_previous_7d"] = (
    transactions_previous_7d
)

In [29]:
average_amount_previous_30d = (transactions.groupby("customer_id", sort = False)
                        .rolling(window = "30D", on = "timestamp", closed = "left")
                        ["amount"].mean().to_numpy()
)

In [30]:
transactions["average_amount_previous_30d"] = (
    average_amount_previous_30d
)

In [31]:
average_amount_previous_90d = (transactions.groupby("customer_id", sort = False)
                        .rolling(window = "90D", on = "timestamp", closed = "left")
                        ["amount"].mean().to_numpy()
)

In [32]:
transactions["average_amount_previous_90d"] = (
    average_amount_previous_90d
)

In [33]:
transactions[
    [
        "customer_id",
        "timestamp",
        "amount",
        "transactions_previous_7d",
        "average_amount_previous_30d",
        "average_amount_previous_90d",
    ]
].head(20)

,customer_id,timestamp,amount,transactions_previous_7d,average_amount_previous_30d,average_amount_previous_90d
0,C00001,2025-01-10 13:13:00,30.51,0,NaN,NaN
1,C00001,2025-01-20 12:41:00,27.45,0,30.510000,30.510000
2,C00001,2025-01-23 09:41:00,105.28,1,28.980000,28.980000
3,C00001,2025-01-29 10:54:00,28.29,1,54.413333,54.413333
4,C00001,2025-02-10 09:47:00,24.60,0,53.673333,47.882500
5,C00001,2025-02-13 09:03:00,56.49,1,46.405000,43.226000
6,C00001,2025-02-16 08:34:00,15.14,2,48.422000,45.436667
7,C00001,2025-02-27 08:44:00,12.31,0,31.130000,41.108571
8,C00001,2025-03-11 10:45:00,27.02,0,27.135000,37.508750
9,C00001,2025-03-27 14:59:00,14.10,0,19.665000,36.343333


In [34]:
"""An operation of EUR500 could be normal for one customer and an outlier for another, so I will use
actual amount / 30d average amount
1.0 means it equals that averge
2.0 it doubles it
5.0 five times such average
"""

previous_30d_denominator = (transactions["average_amount_previous_30d"].replace(0, np.nan))

transactions["amount_ratio_to_previous_30d"] = (transactions["amount"] / previous_30d_denominator)

In [35]:
transactions[
    [
        "transaction_id",
        "customer_id",
        "timestamp",
        "amount",
        "average_amount_previous_30d",
        "amount_ratio_to_previous_30d",
        "is_injected_anomaly",
        "anomaly_type",
    ]
].sort_values(
    "amount_ratio_to_previous_30d",
    ascending=False,
).head()

,transaction_id,customer_id,timestamp,amount,average_amount_previous_30d,amount_ratio_to_previous_30d,is_injected_anomaly,anomaly_type
536,T0000293,C00050,2025-04-08 13:16:00,899.31,30.5500,29.437316,False,none
38,T0000386,C00003,2025-01-16 21:30:00,366.48,23.1700,15.817005,False,none
955,T0000278,C00095,2025-08-27 18:42:00,702.78,48.8625,14.382809,True,high_amount
438,T0000351,C00042,2025-01-27 13:40:00,633.74,45.0500,14.067481,True,high_amount
833,T0000310,C00080,2025-11-23 21:46:00,205.27,15.2200,13.486859,False,none


In [36]:
# Accumulated history
expanding_average = transactions.groupby("customer_id", sort = False)["amount"].expanding(min_periods = 1).mean()
expanding_average_previous = (expanding_average.groupby(level = 0).shift(1).reset_index(level = 0, drop = True).sort_index())

In [37]:
transactions["average_amount_all_previous"] = (
    expanding_average_previous
)

In [38]:
first_customer_id = transactions["customer_id"].iloc[0]

transactions.loc[
    transactions["customer_id"].eq(first_customer_id),
    [
        "customer_id",
        "timestamp",
        "amount",
        "average_amount_all_previous",
    ],
].head()

,customer_id,timestamp,amount,average_amount_all_previous
0,C00001,2025-01-10 13:13:00,30.51,NaN
1,C00001,2025-01-20 12:41:00,27.45,30.510000
2,C00001,2025-01-23 09:41:00,105.28,28.980000
3,C00001,2025-01-29 10:54:00,28.29,54.413333
4,C00001,2025-02-10 09:47:00,24.60,47.882500


In [39]:
# Monthly summary
transactions_with_time_index = (transactions.set_index("timestamp"))
customer_monthly_activity = (
    transactions_with_time_index
    .groupby(
        "customer_id"
    )
    .resample("MS")
    .agg(
        transaction_count=(
            "transaction_id",
            "count",
        ),
        total_amount=(
            "amount",
            "sum",
        ),
        total_effective_amount=(
            "effective_amount",
            "sum",
        ),
        average_amount=(
            "amount",
            "mean",
        ),
        injected_anomalies=(
            "is_injected_anomaly",
            "sum",
        ),
    ).reset_index()
)

In [40]:
customer_monthly_activity = (
    customer_monthly_activity
    .rename(
        columns={
            "timestamp": "month",
        }
    )
)

In [41]:
customer_monthly_activity.head(20)

,customer_id,month,transaction_count,total_amount,total_effective_amount,average_amount,injected_anomalies
0,C00001,2025-01-01,4,191.53,191.53,47.8825,0
1,C00001,2025-02-01,4,108.54,108.54,27.1350,0
2,C00001,2025-03-01,2,41.12,-12.92,20.5600,0
3,C00001,2025-04-01,2,41.22,41.22,20.6100,0
4,C00001,2025-05-01,1,19.23,19.23,19.2300,0
5,C00001,2025-06-01,3,153.45,153.45,51.1500,0
6,C00001,2025-07-01,1,76.71,76.71,76.7100,0
7,C00001,2025-08-01,2,93.63,93.63,46.8150,0
8,C00001,2025-09-01,1,50.53,-50.53,50.5300,0
9,C00001,2025-10-01,1,23.50,23.50,23.5000,0


In [42]:
customer_monthly_activity = (
    customer_monthly_activity
    .sort_values(
        [
            "customer_id",
            "month",
        ]
    )
    .reset_index(
        drop=True
    )
)

In [43]:
customer_monthly_activity["transaction_count_change"] = (
    customer_monthly_activity
    .groupby("customer_id")
    ["transaction_count"]
    .diff()
)

In [44]:
customer_monthly_activity["effective_amount_change"] = (
    customer_monthly_activity
    .groupby("customer_id")
    ["total_effective_amount"]
    .diff()
)

In [45]:
customer_monthly_activity[
    [
        "customer_id",
        "month",
        "transaction_count",
        "transaction_count_change",
        "total_effective_amount",
        "effective_amount_change",
    ]
].head(20)

,customer_id,month,transaction_count,transaction_count_change,total_effective_amount,effective_amount_change
0,C00001,2025-01-01,4,NaN,191.53,NaN
1,C00001,2025-02-01,4,0.0,108.54,-82.99
2,C00001,2025-03-01,2,-2.0,-12.92,-121.46
3,C00001,2025-04-01,2,0.0,41.22,54.14
4,C00001,2025-05-01,1,-1.0,19.23,-21.99
5,C00001,2025-06-01,3,2.0,153.45,134.22
6,C00001,2025-07-01,1,-2.0,76.71,-76.74
7,C00001,2025-08-01,2,1.0,93.63,16.92
8,C00001,2025-09-01,1,-1.0,-50.53,-144.16
9,C00001,2025-10-01,1,0.0,23.50,74.03


In [46]:
# Validation time!

first_transaction_mask = (
    transactions["customer_transaction_number"].eq(1))

temporal_validation = pd.Series(
    {
        "row_count_preserved": (
            len(transactions)
            == 1_000
        ),
        "all_transactions_have_customer": (
            transactions[
                "customer_id"
            ]
            .notna()
            .all()
        ),
        "all_transactions_have_category": (
            transactions[
                "merchant_category"
            ]
            .notna()
            .all()
        ),
        "first_transactions_have_no_previous_timestamp": (
            transactions.loc[
                first_transaction_mask,
                "previous_timestamp",
            ]
            .isna()
            .all()
        ),
        "first_transactions_have_zero_previous_7d": (
            transactions.loc[
                first_transaction_mask,
                "transactions_previous_7d",
            ]
            .eq(0)
            .all()
        ),
        "time_differences_are_non_negative": (
            transactions[
                "hours_since_previous_transaction"
            ]
            .dropna()
            .ge(0)
            .all()
        ),
        "rolling_counts_are_non_negative": (
            transactions[
                "transactions_previous_7d"
            ]
            .ge(0)
            .all()
        ),
        "anomaly_labels_are_preserved": (
            transactions[
                "is_injected_anomaly"
            ]
            .eq(
                transactions[
                    "anomaly_type"
                ]
                .ne("none")
            )
            .all()
        ),
        "monthly_counts_match_transactions": (
            customer_monthly_activity[
                "transaction_count"
            ]
            .sum()
            == len(transactions)
        ),
    },
    name="passed"
)

temporal_validation



row_count_preserved                              True
all_transactions_have_customer                   True
all_transactions_have_category                   True
first_transactions_have_no_previous_timestamp    True
first_transactions_have_zero_previous_7d         True
time_differences_are_non_negative                True
rolling_counts_are_non_negative                  True
anomaly_labels_are_preserved                     True
monthly_counts_match_transactions                True
Name: passed, dtype: bool

In [47]:
# Saving results

synthetic_processed_directory = (
    project_root
    / "data"
    / "synthetic"
    / "processed"
)

synthetic_processed_directory.mkdir(
    parents=True,
    exist_ok=True,
)

In [48]:
temporal_transactions_path = (
    synthetic_processed_directory
    / "synthetic_transactions_temporal_1000.csv"
)

In [49]:
monthly_activity_path = (
    synthetic_processed_directory
    / "synthetic_customer_monthly_activity.csv"
)

In [50]:
transactions.to_csv(
    temporal_transactions_path,
    index=False,
)

In [51]:
customer_monthly_activity.to_csv(
    monthly_activity_path,
    index=False,
)

In [52]:
pd.Series(
    {
        "temporal_transactions": (
            temporal_transactions_path.exists()
        ),
        "monthly_activity": (
            monthly_activity_path.exists()
        ),
    }
)

temporal_transactions    True
monthly_activity         True
dtype: bool

## Temporal feature results

The synthetic transactions were ordered by customer and timestamp and the following historical features were created:

- Transaction number within each customer
- Previous transaction timestamp
- Previous amount and merchant category
- Hours and days since the previous transaction
- Number of previous transactions in seven days
- Average amount during the previous 30 days
- Average amount during the previous 90 days
- Ratio between the current amount and the previous 30 day average
- Average amount across all previous customer history
- Monthly customer activity and monthly changes

The current transaction was excluded from the rolling and expanding historical statistics to prevent information leakage. I will use these variables in the next phase to detect high amounts, unusually frequent activity and deviations from customer behaviour